# 📊 Unidad 3: Modelado de Datos y Machine Learning
## Laboratorio (Herramientas) - Universidad del Aconcagua
### Contenido Teórico

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta unidad teórica, serás capaz de:

1. ✅ Comprender Delta Lake y sus ventajas (ACID, time travel, schema evolution)
2. ✅ Dominar agregaciones avanzadas y window functions en SQL/PySpark
3. ✅ Aplicar técnicas de feature engineering para machine learning
4. ✅ Entender el flujo completo de un proyecto de ML en Databricks
5. ✅ Conocer métricas de evaluación para modelos de clasificación y regresión
6. ✅ Implementar mejores prácticas de modelado y experimentación

---

### 📚 Contenido

1. Delta Lake: La Capa de Datos Confiable
2. Agregaciones y Window Functions
3. Feature Engineering: Creando Variables Predictivas
4. Flujo de Machine Learning en Databricks
5. Métricas de Evaluación de Modelos
6. Mejores Prácticas en Modelado

---

### ⏱️ Duración Estimada: 2.5 horas

## 1️⃣ Delta Lake: La Capa de Datos Confiable

### ¿Qué es Delta Lake?

**Delta Lake** es una capa de almacenamiento open-source que aporta confiabilidad a data lakes:

* Construido sobre Apache Parquet
* Formato estándar en Databricks
* Compatible con Spark API

### Características Principales

#### 🔒 **Transacciones ACID**

Delta Lake garantiza:
* **Atomicidad**: Operación completa o nada
* **Consistencia**: Siempre un estado válido
* **Aislamiento**: Lecturas consistentes durante escrituras
* **Durabilidad**: Cambios permanentes

```python
# Escritura con transacciones ACID
df.write.format("delta").mode("append").save("/path/to/delta-table")
```

#### ⏮️ **Time Travel (Versionado de Datos)**

Accede a versiones históricas de tus datos:

```python
# Leer versión anterior
df_v1 = spark.read.format("delta").option("versionAsOf", 1).load("/path")

# Leer por timestamp
df_yesterday = spark.read.format("delta") \
    .option("timestampAsOf", "2024-01-15") \
    .load("/path")
```

**Casos de uso:**
* Auditoría y compliance
* Reproducibilidad de análisis
* Rollback de cambios erróneos
* Comparación temporal

#### 🔄 **Schema Evolution**

Evoluciona el schema sin romper pipelines:

```python
# Agregar columnas automáticamente
df_new.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save("/path")
```

#### ⚡ **Optimizaciones de Performance**

1. **Z-Ordering**: Co-localiza datos relacionados
```sql
OPTIMIZE tabla ZORDER BY (columna_filtro_frecuente)
```

2. **Data Skipping**: Salta archivos irrelevantes
3. **Compaction**: Combina archivos pequeños
```sql
OPTIMIZE tabla
```

#### 🗑️ **VACUUM: Limpieza de Archivos Antiguos**

```sql
-- Eliminar archivos no referenciados (>7 días)
VACUUM tabla RETAIN 168 HOURS
```

⚠️ **Cuidado**: No ejecutes VACUUM si necesitas time travel extenso.

## 2️⃣ Agregaciones y Window Functions

### Agregaciones Avanzadas

#### **GROUP BY con Múltiples Dimensiones**

```sql
SELECT 
    categoria,
    YEAR(fecha) AS anio,
    MONTH(fecha) AS mes,
    COUNT(*) AS total_ventas,
    SUM(monto) AS ingresos_totales,
    AVG(monto) AS ticket_promedio,
    MAX(monto) AS venta_maxima
FROM ventas
GROUP BY categoria, YEAR(fecha), MONTH(fecha)
ORDER BY anio, mes, ingresos_totales DESC
```

#### **HAVING: Filtrar Después de Agrupar**

```sql
SELECT 
    producto,
    SUM(cantidad) AS total_vendido
FROM ventas
GROUP BY producto
HAVING SUM(cantidad) > 1000  -- Filtro post-agregación
ORDER BY total_vendido DESC
```

### Window Functions (Funciones de Ventana)

**Window functions** permiten cálculos sobre un "marco" de filas relacionadas **sin colapsar** los datos.

#### **Estructura Básica**

```sql
funcion_agregada() OVER (
    PARTITION BY columna_grupo
    ORDER BY columna_orden
    ROWS/RANGE especificacion_ventana
)
```

#### **Ejemplos Prácticos**

**1. Ranking dentro de grupos:**
```sql
SELECT 
    producto,
    categoria,
    ventas,
    ROW_NUMBER() OVER (PARTITION BY categoria ORDER BY ventas DESC) AS ranking,
    RANK() OVER (PARTITION BY categoria ORDER BY ventas DESC) AS rank_con_empates
FROM productos_ventas
```

**2. Acumulados (Running Totals):**
```sql
SELECT 
    fecha,
    ventas_dia,
    SUM(ventas_dia) OVER (ORDER BY fecha) AS ventas_acumuladas,
    AVG(ventas_dia) OVER (ORDER BY fecha ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS promedio_7dias
FROM ventas_diarias
```

**3. Comparaciones con periodo anterior (Lag/Lead):**
```sql
SELECT 
    fecha,
    ingresos,
    LAG(ingresos, 1) OVER (ORDER BY fecha) AS ingresos_dia_anterior,
    ingresos - LAG(ingresos, 1) OVER (ORDER BY fecha) AS diferencia_dia_anterior,
    LEAD(ingresos, 1) OVER (ORDER BY fecha) AS ingresos_dia_siguiente
FROM ingresos_diarios
```

#### **En PySpark**

```python
from pyspark.sql import Window
from pyspark.sql.functions import row_number, rank, sum, avg, lag

# Definir ventana
window_spec = Window.partitionBy("categoria").orderBy(col("ventas").desc())

# Aplicar función de ventana
df_ranked = df.withColumn("ranking", row_number().over(window_spec))
```

## 3️⃣ Feature Engineering: Creando Variables Predictivas

### ¿Qué es Feature Engineering?

**Feature engineering** es el proceso de crear, transformar y seleccionar variables (features) que mejoran el rendimiento de modelos de ML.

> "Los datos crudos son solo el inicio; las features son la clave."

### Tipos de Features

#### 1️⃣ **Features Temporales**

Extrae información de fechas:

```python
from pyspark.sql.functions import year, month, dayofweek, hour, quarter

df = df.withColumn("anio", year("fecha")) \
       .withColumn("mes", month("fecha")) \
       .withColumn("dia_semana", dayofweek("fecha")) \
       .withColumn("trimestre", quarter("fecha")) \
       .withColumn("es_fin_semana", when(dayofweek("fecha").isin([1, 7]), 1).otherwise(0))
```

**Para qué sirven:**
* Capturar estacionalidad
* Identificar patrones cíclicos
* Diferenciar comportamiento por periodo

#### 2️⃣ **Encoding de Variables Categóricas**

**Label Encoding** (ordinal):
```python
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol="categoria", outputCol="categoria_idx")
df = indexer.fit(df).transform(df)
```

**One-Hot Encoding** (nominal):
```python
from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder(inputCols=["categoria_idx"], outputCols=["categoria_vec"])
df = encoder.fit(df).transform(df)
```

#### 3️⃣ **Escalado y Normalización**

**StandardScaler** (media 0, std 1):
```python
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(inputCol="features", outputCol="scaled_features", 
                        withMean=True, withStd=True)
df_scaled = scaler.fit(df).transform(df)
```

**MinMaxScaler** (rango 0-1):
```python
from pyspark.ml.feature import MinMaxScaler

scaler = MinMaxScaler(inputCol="features", outputCol="scaled_features")
```

#### 4️⃣ **Agregaciones como Features**

```python
# Features agregadas por cliente
features_cliente = df.groupBy("cliente_id").agg(
    count("*").alias("total_compras"),
    sum("monto").alias("gasto_total"),
    avg("monto").alias("ticket_promedio"),
    max("fecha").alias("ultima_compra"),
    countDistinct("producto").alias("variedad_productos")
)
```

#### 5️⃣ **Features de Interacción**

Combina variables existentes:

```python
# Interacciones multiplicativas
df = df.withColumn("precio_por_cantidad", col("precio") * col("cantidad"))

# Ratios
df = df.withColumn("ratio_descuento", col("descuento") / col("precio_original"))

# Binning (discretización)
df = df.withColumn("segmento_edad", 
    when(col("edad") < 25, "joven")
    .when(col("edad") < 50, "adulto")
    .otherwise("senior")
)
```

### Mejores Prácticas de Feature Engineering

✅ **Entiende el dominio**: La creatividad en features viene del conocimiento del negocio  
✅ **Evita data leakage**: No uses información del futuro para predecir el pasado  
✅ **Maneja valores faltantes**: Imputa o crea flag `is_missing`  
✅ **Documenta features**: Qué significa cada variable y cómo se calculó  
✅ **Valida en test set**: Asegura que features generalizan  

## 4️⃣ Flujo de Machine Learning en Databricks

### El Ciclo Completo de ML

```
1. PREPARACIÓN DE DATOS
   ↓
2. FEATURE ENGINEERING
   ↓
3. ENTRENAMIENTO DE MODELOS
   ↓
4. EVALUACIÓN Y SELECCIÓN
   ↓
5. DEPLOYMENT
   ↓
6. MONITOREO
```

### 1. Preparación de Datos

```python
# Cargar desde Delta Lake
df = spark.read.format("delta").load("/path/to/table")

# Limpieza
df = df.dropna(subset=["columna_critica"])
df = df.filter(col("valor") > 0)

# Split train/test
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
```

### 2. Feature Engineering (ver sección anterior)

### 3. Entrenamiento con MLlib

```python
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline

# Ensamblar features
assembler = VectorAssembler(
    inputCols=["feature1", "feature2", "feature3"],
    outputCol="features"
)

# Modelo
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="target",
    numTrees=100,
    maxDepth=10
)

# Pipeline
pipeline = Pipeline(stages=[assembler, rf])
model = pipeline.fit(train_df)
```

### 4. Predicción y Evaluación

```python
# Predicciones
predictions = model.transform(test_df)

# Evaluación
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="target",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator.evaluate(predictions)
print(f"RMSE: {rmse}")
```

### 5. Tracking con MLflow

```python
import mlflow
import mlflow.spark

with mlflow.start_run(run_name="random_forest_v1"):
    # Log parámetros
    mlflow.log_param("num_trees", 100)
    mlflow.log_param("max_depth", 10)
    
    # Entrenar
    model = pipeline.fit(train_df)
    
    # Evaluar y log métricas
    predictions = model.transform(test_df)
    rmse = evaluator.evaluate(predictions)
    mlflow.log_metric("rmse", rmse)
    
    # Log modelo
    mlflow.spark.log_model(model, "model")
```

### 6. Registro y Deployment

```python
# Registrar modelo en Unity Catalog
mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name="main.ml_models.prediccion_ventas"
)

# Cargar modelo registrado
model_uri = "models:/main.ml_models.prediccion_ventas/1"
loaded_model = mlflow.spark.load_model(model_uri)
```

## 5️⃣ Métricas de Evaluación de Modelos

### Métricas para Regresión

#### **MAE (Mean Absolute Error)**
* Promedio de errores absolutos
* **Interpretación**: Error promedio en las mismas unidades del target
* **Ventaja**: Fácil de entender
* **Fórmula**: `MAE = mean(|y_real - y_pred|)`

#### **RMSE (Root Mean Squared Error)**
* Raíz cuadrada del promedio de errores al cuadrado
* **Interpretación**: Penaliza más los errores grandes
* **Cuándo usar**: Cuando errores grandes son costosos
* **Fórmula**: `RMSE = sqrt(mean((y_real - y_pred)²))`

#### **R² (Coeficiente de Determinación)**
* Proporción de varianza explicada por el modelo
* **Rango**: 0 a 1 (o negativo si el modelo es muy malo)
* **Interpretación**: R² = 0.85 → modelo explica 85% de la variabilidad
* **Fórmula**: `R² = 1 - (SS_res / SS_tot)`

#### **MAPE (Mean Absolute Percentage Error)**
* Error porcentual promedio
* **Ventaja**: Independiente de escala
* **Cuidado**: Indefinido si y_real = 0
* **Fórmula**: `MAPE = mean(|y_real - y_pred| / |y_real|) * 100`

### Métricas para Clasificación

#### **Matriz de Confusión**

```
                    Predicho
                 Positivo  Negativo
Real  Positivo     TP        FN
      Negativo     FP        TN
```

* **TP** (True Positive): Predicho + y es +
* **TN** (True Negative): Predicho - y es -
* **FP** (False Positive): Predicho + pero es - (Error Tipo I)
* **FN** (False Negative): Predicho - pero es + (Error Tipo II)

#### **Accuracy (Exactitud)**
* Proporción de predicciones correctas
* **Fórmula**: `(TP + TN) / Total`
* **Problema**: Engañosa con clases desbalanceadas

#### **Precision (Precisión)**
* De los que predije como positivos, ¿cuántos lo son realmente?
* **Fórmula**: `TP / (TP + FP)`
* **Cuándo importa**: Minimizar falsos positivos (spam detection)

#### **Recall (Sensibilidad / Exhaustividad)**
* De los positivos reales, ¿cuántos detecté?
* **Fórmula**: `TP / (TP + FN)`
* **Cuándo importa**: Minimizar falsos negativos (detección de fraude, diagnósticos médicos)

#### **F1-Score**
* Media armónica de Precision y Recall
* **Fórmula**: `2 * (Precision * Recall) / (Precision + Recall)`
* **Cuándo usar**: Balance entre precision y recall

#### **AUC-ROC (Area Under Curve - Receiver Operating Characteristic)**
* Mide capacidad de discriminación del modelo
* **Rango**: 0.5 (random) a 1.0 (perfecto)
* **Interpretación**: Probabilidad de que el modelo ranquee un positivo real más alto que un negativo real
* **Ventaja**: Independiente del threshold

### ¿Qué Métrica Usar?

| Escenario | Métrica Recomendada |
|-----------|---------------------|
| Regresión general | RMSE o MAE |
| Regresión con outliers sensibles | MAE (más robusto) |
| Comparar modelos entre escalas | MAPE o R² |
| Clasificación balanceada | Accuracy, F1 |
| Clasificación desbalanceada | F1, Precision/Recall, AUC-ROC |
| Costo alto de FP | Precision |
| Costo alto de FN | Recall |
| Ranking (ej: recomendaciones) | AUC-ROC |

## 6️⃣ Mejores Prácticas en Modelado

### 1. División de Datos

#### **Train / Validation / Test Split**

```python
# Split típico: 70% train, 15% validation, 15% test
train, temp = df.randomSplit([0.7, 0.3], seed=42)
validation, test = temp.randomSplit([0.5, 0.5], seed=42)
```

**Usos:**
* **Train**: Entrenar el modelo
* **Validation**: Tunear hiperparámetros, early stopping
* **Test**: Evaluación final (¡solo una vez!)

#### **Cross-Validation**

```python
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

# Grid de hiperparámetros
paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [50, 100, 200]) \
    .addGrid(rf.maxDepth, [5, 10, 15]) \
    .build()

# Cross-validator
crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(labelCol="target", metricName="rmse"),
    numFolds=5
)

cvModel = crossval.fit(train)
best_model = cvModel.bestModel
```

### 2. Prevenir Overfitting

**Síntomas de overfitting:**
* Alta performance en train, baja en test
* Modelo muy complejo
* Memoriza patrones aleatorios

**Soluciones:**

1. **Más datos** (si es posible)
2. **Regularización** (L1, L2)
3. **Reducción de complejidad** (menos features, árboles menos profundos)
4. **Early stopping** (detener antes de sobreajustar)
5. **Cross-validation** (validación robusta)
6. **Ensemble methods** (promedio de múltiples modelos)

### 3. Manejo de Desbalanceo de Clases

```python
# 1. Sobremuestreo de clase minoritaria
minority_class = df.filter(col("target") == 1)
majority_class = df.filter(col("target") == 0)

minority_oversampled = minority_class.sample(withReplacement=True, fraction=3.0)
balanced_df = majority_class.union(minority_oversampled)

# 2. Pesos por clase
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(weightCol="classWeights")

# 3. Cambiar métrica de evaluación (usar F1 o AUC-ROC en lugar de Accuracy)
```

### 4. Feature Importance

```python
# Obtener importancia de features
rf_model = model.stages[-1]  # Último stage del pipeline
feature_importances = rf_model.featureImportances

# Visualizar
import pandas as pd
import matplotlib.pyplot as plt

features_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importances.toArray()
}).sort_values('importance', ascending=False)

plt.barh(features_df['feature'][:10], features_df['importance'][:10])
plt.xlabel('Importance')
plt.title('Top 10 Features')
plt.show()
```

### 5. Reproducibilidad

✅ **Fija seeds aleatorios**: `randomSplit(..., seed=42)`  
✅ **Versiona datos**: Delta Lake time travel  
✅ **Registra hiperparámetros**: MLflow tracking  
✅ **Documenta transformaciones**: Comentarios y metadata  
✅ **Guarda modelos**: MLflow Model Registry  

### 6. Monitoreo Post-Deployment

* **Data drift**: ¿Los datos nuevos son similares a los de entrenamiento?
* **Concept drift**: ¿La relación input-output cambió?
* **Performance decay**: ¿Las métricas se degradan con el tiempo?
* **Reentrenamiento**: Programa re-entrenamientos periódicos

## 📚 Resumen y Próximos Pasos

### ✅ Conceptos Clave Aprendidos

1. **Delta Lake**: ACID, time travel, schema evolution, optimizaciones
2. **Agregaciones avanzadas**: GROUP BY, HAVING, múltiples dimensiones
3. **Window functions**: Ranking, acumulados, LAG/LEAD, particiones
4. **Feature engineering**: Temporales, encoding, escalado, agregaciones, interacciones
5. **Flujo ML**: Preparación → Features → Entrenamiento → Evaluación → Deployment
6. **Métricas**: RMSE/MAE/R² (regresión), Precision/Recall/F1/AUC-ROC (clasificación)
7. **Mejores prácticas**: Train/val/test split, cross-validation, prevención de overfitting

### 🚀 Preparación para el TP03

En el trabajo práctico aplicarás:
* Agregaciones y window functions sobre datos reales
* Feature engineering creativo
* Entrenamiento de modelo de ML
* Evaluación con múltiples métricas
* Tracking con MLflow
* Registro del mejor modelo

### 📖 Recursos Adicionales

* [Delta Lake Documentation](https://docs.delta.io/)
* [PySpark MLlib Guide](https://spark.apache.org/docs/latest/ml-guide.html)
* [MLflow Documentation](https://mlflow.org/docs/latest/index.html)
* [Feature Engineering Book (Alice Zheng)](https://www.oreilly.com/library/view/feature-engineering-for/9781491953235/)

---

### 💬 Preguntas de Reflexión

1. ¿Cuándo usarías una window function en lugar de un GROUP BY?
2. ¿Qué features crearías para predecir la demanda de un producto?
3. ¿Por qué es peligroso evaluar el modelo solo con Accuracy en datos desbalanceados?

---

**¡Listo para construir modelos predictivos en el TP03! 🤖📈**